Ten skrypt tworzy agenta do automatyzacji przeglądarki internetowej, wykorzystując biblioteki Playwright oraz platformę Together AI do interakcji z modelami językowymi.

# Setup

In [ ]:
!uv pip install -qU together playwright pytest-playwright

In [6]:
!ls -lh

total 940K
drwxr-xr-x 1 root root 4.0K Apr 29 13:36 sample_data
-rw-r--r-- 1 root root 936K May  1 14:18 sample_screenshot.png


*   `together` to narzędzie do integracji z usługami sztucznej inteligencji - pozwala na łatwe wykorzystanie różnych modeli AI w projektach. Jest szczególnie przydatna przy pracy z generatywnymi modelami tekstu.
*   `Playwright` to framework do automatyzacji przeglądarek internetowych. Umożliwia programowe sterowanie przeglądarkami jak Chrome, Firefox czy Safari. Dzięki niemu można pisać skrypty, które wykonują działania w przeglądarce - wypełniają formularze, klikają przyciski, pobierają dane ze stron internetowych.

In [9]:
!playwright install
!playwright install chrome

167.7 MiB [] 0% 0.0s167.7 MiB [] 0% 4.4s167.7 MiB [] 1% 2.0s167.7 MiB [] 2% 1.8s167.7 MiB [] 3% 1.8s167.7 MiB [] 4% 1.9s167.7 MiB [] 4% 2.1s167.7 MiB [] 5% 2.0s167.7 MiB [] 6% 2.1s167.7 MiB [] 7% 2.0s167.7 MiB [] 8% 2.0s167.7 MiB [] 9% 1.9s167.7 MiB [] 10% 1.7s167.7 MiB [] 12% 1.5s167.7 MiB [] 13% 1.5s167.7 MiB [] 14% 1.5s167.7 MiB [] 15% 1.5s167.7 MiB [] 17% 1.4s167.7 MiB [] 18% 1.3s167.7 MiB [] 20% 1.2s167.7 MiB [] 21% 1.2s167.7 MiB [] 22% 1.2s167.7 MiB [] 23% 1.2s167.7 MiB [] 25% 1.1s167.7 MiB [] 27% 1.1s167.7 MiB [] 29% 1.0s167.7 MiB [] 30% 1.0s167.7 MiB [] 31% 1.0s167.7 MiB [] 32% 0.9s167.7 MiB [] 33% 0.9s167.7 MiB [] 35% 0.9s167.7 MiB [] 37% 0.9s167.7 MiB [] 38% 0.8s167.7 MiB [] 40% 0.8s167.7 MiB [] 41% 0.8s167.7 MiB [] 43% 0.8s167.7 MiB [] 44% 0.7s167.7 MiB [] 45% 0.7s167.7 MiB [] 46% 0.7s167.7 MiB [] 47% 0.7s167.7 MiB [] 48% 0.7s167.7 MiB [] 50% 0.7s167.7 MiB [] 52% 0.7s167.7 MiB [] 53% 0.7s167.7 MiB [] 54% 0.6s167.7 MiB [] 56% 0.6s167.7 MiB [] 57% 0.6s167.7 MiB [] 58% 0.6s167.

In [ ]:
from together import Together
import base64
from playwright.async_api import async_playwright, Page
import asyncio
import json
import re
from google.colab import userdata

Ten kod jest skryptem w Pythonie, który łączy kilka bibliotek do pracy z modelami AI oraz automatyzacji przeglądarki internetowej.

Kod importuje następujące biblioteki:
- `Together` - klient API dla platformy Together.ai, która udostępnia różne modele AI
- `base64` - służy do kodowania i dekodowania danych w formacie base64
- `async_playwright` z pakietu playwright - narzędzie do automatyzacji przeglądarki internetowej w trybie asynchronicznym
- `asyncio` - biblioteka do pisania kodu asynchronicznego w Pythonie
- `json` - obsługuje operacje na danych w formacie JSON
- `re` - biblioteka do pracy z wyrażeniami regularnymi


In [ ]:
class CFG:
    model = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

In [ ]:
client = Together(api_key=userdata.get("together"))

# Funkcje

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

Funkcja `encode_image` służy do kodowania obrazu w formę tekstową przy użyciu kodowania Base64. Jest to niezwykle użyteczna technika, gdy potrzebujemy przesłać obraz przez interfejsy, które przyjmują tylko tekst, takie jak wiele API.

Przeanalizujmy działanie funkcji krok po kroku:

1. Funkcja przyjmuje jeden parametr: `image_path`, który jest ścieżką do pliku obrazu na dysku.

2. Instrukcja `with open(image_path, "rb") as image_file:` otwiera plik obrazu w trybie binarnym (oznaczonym przez `"rb"`). Tryb binarny jest kluczowy, ponieważ obrazy to pliki binarne, nie tekstowe. Użycie konstrukcji `with` zapewnia automatyczne zamknięcie pliku po zakończeniu operacji, co jest dobrą praktyką programistyczną.

3. Wewnątrz bloku `with`, wyrażenie `image_file.read()` wczytuje całą zawartość pliku obrazu jako ciąg bajtów.

4. Następnie `base64.b64encode()` przekształca te surowe dane binarne na ich reprezentację Base64, która składa się tylko z drukowalnych znaków ASCII. Kodowanie Base64 zamienia każde 3 bajty danych binarnych na 4 znaki tekstowe, używając tylko 64 różnych znaków (stąd nazwa).

5. Na końcu, metoda `.decode('utf-8')` konwertuje zakodowane bajty Base64 na zwykły string UTF-8, który można łatwo przesyłać jako tekst.

Proces ten jest analogiczny do tłumaczenia książki na inny język - oryginalna treść (dane binarne obrazu) zostaje przekształcona w inną formę (tekst Base64), zachowując dokładnie te same informacje, ale w formacie, który może być używany w środowiskach akceptujących tylko tekst.

In [ ]:
async def fill_date(page: Page, selector: str, date_str: str):
    """Fills a date input field with a given date string."""
    # Wait for the element to be ready
    await page.wait_for_selector(selector)
    
    # Focus and fill the input
    await page.click(selector, click_count=3)  # Select all text if any
    await page.fill(selector, date_str)

In [ ]:
def parse_accessibility_tree(node, indent=0):
    """
    Recursively parses the accessibility tree and prints a readable structure.
    Args:
        node (dict): A node in the accessibility tree.
        indent (int): Indentation level for the nested structure.
    """
    # Initialize res as an empty string at the start of each parse
    res = ""

    def _parse_node(node, indent, res):
        # Base case: If the node is empty or doesn't have a 'role', skip it
        if not node or "role" not in node:
            return res

        # Indentation for nested levels
        indented_space = " " * indent

        # Add node's name and role to result string
        if "value" in node:
            res = (
                res
                + f"{indented_space}Role: {node['role']} - Name: {node.get('name', 'No name')} - Value: {node['value']}\n"
            )
        else:
            res = (
                res
                + f"{indented_space}Role: {node['role']} - Name: {node.get('name', 'No name')}\n"
            )

        # If the node has children, recursively parse them
        if "children" in node:
            for child in node["children"]:
                res = _parse_node(
                    child, indent + 2, res
                )  # Increase indentation for child nodes

        return res

    return _parse_node(node, indent, res)

Ta funkcja `parse_accessibility_tree` to narzędzie służące do analizy i prezentacji drzewa dostępności w czytelnej formie. Drzewa dostępności są używane w interfejsach użytkownika do reprezentowania struktury elementów na ekranie w sposób, który może być interpretowany przez technologie asystujące, jak czytniki ekranu.

# Prompty

In [ ]:
planning_prompt = """
Given a user request, define a very simple plan of subtasks (actions) to achieve the desired outcome and execute them iteratively using Playwright.

1. Understand the Task:
   - Interpret the user's request and identify the core goal.
   - Break down the task into a few smaller, actionable subtasks to achieve the goal effectively.

2. Planning Actions:
   - Translate the user's request into a high-level plan of actions.
   - Example actions include:
     - Searching for specific information.
     - Navigating to specified URLs.
     - Interacting with website elements (clicking, filling).
     - Extracting or validating data.

Input:
- User Request (Task)

Output from the Agent:
- Step-by-Step Action Plan:: Return only an ordered list of actions. Only return the list, no other text.

**Example User Requests and Agent Behavior:**

1. **Input:** "Search for a product on Amazon."
   - **Output:**
     1. Navigate to Amazon's homepage.
     2. Enter the product name in the search bar and perform the search.
     3. Extract and display the top results, including the product title, price, and ratings.

2. **Input:** "Find the cheapest flight to Tokyo."
   - **Output:**
     1. Visit a flight aggregator website (e.g. Kayak).
     2. Enter the departure city.
     3. Enter the destination city
     4. Enter the start and end dates.
     5. Extract and compare the flight options, highlighting the cheapest option.

3. **Input:** "Buy tickets for the next Warriors game."
   - **Output:**
     1. Navigate to a ticket-selling platform (e.g., Ticketmaster).
     2. Fill the search bar with the team name.
     2. Search for upcoming team games.
     3. Select the next available game and purchase tickets for the specified quantity.
"""

In [16]:
execution_prompt = """
You will be given a task, a website's page accessibility tree, and the page screenshot as context. The screenshot is where you are now, use it to understand the accessibility tree. Based on that information, you need to decide the next step action. ONLY RETURN THE NEXT STEP ACTION IN A SINGLE JSON.

When selecting elements, use elements from the accessibility tree.

Reflect on what you are seeing in the accessibility tree and the screenshot and decide the next step action, elaborate on it in reasoning, and choose the next appropriate action.

Selectors must follow the format:
- For a button with a specific name: "button=ButtonName"
- For a placeholder (e.g., input field): "placeholder=PlaceholderText"
- For text: "text=VisibleText"

Make sure to analyze the accessibility tree and the screenshot to understand the current state, if something is not clear, you can use the previous actions to understand the current state. Explain why you are in the current state in current_state.

You will be given a task and you MUST return the next step action in JSON format:
{
    "current_state": "Where are you now? Analyze the accessibility tree and the screenshot to understand the current state.",
    "reasoning": "What is the next step to accomplish the task?",
    "action": "navigation" or "click" or "fill" or "finished",
    "url": "https://www.example.com", // Only for navigation actions
    "selector": "button=Click me", // For click or fill actions, derived from the accessibility tree
    "value": "Input text", // Only for fill actions
}

### Guidelines:
1. Use **"navigation"** for navigating to a new website through a URL.
2. Use **"click"** for interacting with clickable elements. Examples:
   - Buttons: "button=Click me"
   - Text: "text=VisibleText"
   - Placeholders: "placeholder=Search..."
   - Link: "link=BUY NOW"
3. Use **"fill"** for inputting text into editable fields. Examples:
   - Placeholder: "placeholder=Search..."
   - Textbox: "textbox=Flight destination output"
   - Input: "input=Search..."
4. Use **"finished"** when the task is done. For example:
   - If a task is successfully completed.
   - If navigation confirms you are on the correct page.


### Accessibility Tree Examples:

You will be given an accessibility tree to interact with the webpage. It consists of a nested node structure that represents elements on the page. For example:

Role: generic - Name:
   Role: text - Name: San Francisco (SFO)
   Role: button - Name:
   Role: listitem - Name:
   Role: textbox - Name: Flight origin input
Role: button - Name: Swap departure airport and destination airport
Role: generic - Name:
   Role: textbox - Name: Flight destination input
Role: button - Name: Start date
Role: button - Name:
Role: button - Name:
Role: button - Name: End date
Role: button - Name:
Role: button - Name:
Role: button - Name: Search

This section indicates that there is a textbox with a name "Flight destination input" filled with San Francisco (SFO). There is also a button with the name "Swap departure airport and destination airport". Another textbox with the name "Flight destination input" not filled with any text. There are also buttons with the names "Start date", "End date", which are not filled with any dates, and a button named "Search".

Retry actions at most 2 times before trying a different action.

### Examples:
1. To click on a button labeled "Search":
   {
       "current_state": "On the homepage of a search engine.",
       "reasoning": "The accessibility tree shows a button named 'Search'. Clicking it is the appropriate next step to proceed with the task.",
       "action": "click",
       "selector": "button=Search"
   }

2. To fill a search bar with the text "AI tools":
   {
       "current_state": "On the search page with a focused search bar.",
       "reasoning": "The accessibility tree shows an input field with placeholder 'Search...'. Entering the query 'AI tools' fulfills the next step of the task.",
       "action": "fill",
       "selector": "placeholder=Search...",
       "value": "AI tools"
   }

3. To navigate to a specific URL:
   {
       "current_state": "Starting from a blank page.",
       "reasoning": "The task requires visiting a specific website to gather relevant information. Navigating to the URL is the first step.",
       "action": "navigation",
       "url": "https://example.com"
   }

4. To finish the task:
   {
       "current_state": "Completed the search and extracted the necessary data.",
       "reasoning": "The task goal has been achieved, and no further actions are required.",
       "action": "finished"
   }
"""

In [17]:
# Przykład 1
few_shot_example_1 = """
User Input: "What are the best tacos in San Francisco?"

Agent Step Sequence:
Step 1:
{
    "current_state": "On a blank page.",
    "reasoning": "The task is to find the best tacos in San Francisco, so the first step is to navigate to Google to perform a search.",
    "action": "navigation",
    "url": "https://www.google.com",
}

Step 2:
{
    "current_state": "On the Google homepage.",
    "reasoning": "To search for the best tacos in San Francisco, I need to fill the Google search bar with the query.",
    "action": "fill",
    "selector": "combobox=Search",
    "value": "Best tacos in San Francisco"
}

Step 3:
{
    "current_state": "On Google search results page.",
    "reasoning": "After entering the query, I need to click the search button to retrieve the results.",
    "action": "click",
    "selector": "button=Google Search"
}

Step 4:
{
    "current_state": "On the search results page with multiple links.",
    "reasoning": "From the search results, I need to click on a reliable food-review or blogwebsite link.",
    "action": "click",
    "selector": "text=Yelp"
}

Step 5:
{
    "current_state": "On Yelp's best taqueria near San Francisco page.",
    "reasoning": "The task is complete as I have found the top taquerias in San Francisco.",
    "action": "finished",
    "summary": "I have successfully found the best tacos in San Francisco."
}
"""


In [18]:
# Przykład 2
few_shot_example_2 = """
User Input: Can you send an email to reschedule a meeting for Dmitry at gmail.com for tomorrow morning? I'm sick today.

Agent Step Sequence:
Step 1:
{
    "current_state": "On a blank page.",
    "reasoning": "To send an email, the first step is to navigate to Gmail.",
    "action": "navigation",
    "url": "https://mail.google.com",
}

Step 2:
{
    "current_state": "On Gmail's homepage.",
    "reasoning": "Click the 'Compose' button to start drafting a new email.",
    "action": "click",
    "selector": "button=Compose"
}

Step 3:
{
    "current_state": "In the new email draft window.",
    "reasoning": "Enter Dmitry's email address in the recipient field.",
    "action": "fill",
    "selector": "placeholder=Recipients",
    "value": "dmitry@gmail.com"
}

Step 4:
{
    "current_state": "In the new email draft with the recipient filled.",
    "reasoning": "Set the subject line to indicate the purpose of the email.",
    "action": "fill",
    "selector": "placeholder=Subject",
    "value": "Rescheduling Meeting"
}

Step 5:
{
    "current_state": "In the new email draft with the subject set.",
    "reasoning": "Compose the email body to politely inform Dmitry about rescheduling the meeting.",
    "action": "fill",
    "selector": "placeholder=Email body",
    "value": "Hi Dmitry,\\n\\nI'm feeling unwell today and would like to reschedule our meeting for tomorrow morning. Please let me know if this works for you.\\n\\nBest regards,\\n[Your Name]"
}

Step 6:
{
    "current_state": "In the new email draft with the body composed.",
    "reasoning": "Click the 'Send' button to deliver the email to Dmitry.",
    "action": "click",
    "selector": "button=Send"
}

Step 7:
{
    "current_state": "On Gmail's homepage after sending the email.",
    "reasoning": "The email has been drafted and sent, fulfilling the task of informing Dmitry about the reschedule.",
    "action": "finished",
    "summary": "Email sent to Dmitry to reschedule the meeting for tomorrow morning."
}
"""


In [19]:
few_shot_example_3 = """
User Input: "Find the round trip cheapest flight to Madrid."

Agent Step Sequence:

Step 1:
{
    "current_state": "On a flight booking website.",
    "reasoning": "The task is to find the cheapest round trip flight to Madrid, so the first step is to navigate to a flight aggregator website.",
    "action": "navigation",
    "url": "https://www.example-flight-aggregator.com",
}

Step 2:
{
    "current_state": "On the flight aggregator homepage.",
    "reasoning": "To find flights, I need to fill the departure city field.",
    "action": "fill",
    "selector": "placeholder=Departure City",
    "value": "Your City"
}

Step 3:
{
    "current_state": "On the flight search page with departure city filled.",
    "reasoning": "Fill the destination city field with 'Madrid'.",
    "action": "fill",
    "selector": "placeholder=Where to?",
    "value": "Madrid"
}

Step 4:
{
    "current_state": "On the flight search page with destination city filled.",
    "reasoning": "Fill the Departure field with a date in future",
    "action": "fill",
    "selector": "placeholder=outband_date",
    "value": "2025-10-15",

Step 5:
{
    "current_state": "Departure date filled.",
    "reasoning": "Fill the return field with a date in future after the departure date",
    "action": "fill",
    "selector": "placeholder=return_date",
    "value": '2025-12-08',
}

Step 6:
{
    "current_state": "Return date filled.",
    "reasoning": "Click the 'Done' button.",
    "action": "click",
    "selector": "button=Done"
}

Step 7:
{
    "current_state": "Done button clicked.",
    "reasoning": "Click the 'Search' button to search the flights.",
    "action": "click",
    "selector": "button=Search"
}

Step 8:
{
    "current_state": "Flight options are displayed.",
    "reasoning": "Extract and compare the flight options, highlighting the cheapest option.",
    "action": "finished",
    "summary": "Cheapest round trip flight to Madrid found and displayed."
}
"""

In [20]:
few_shot_examples = [few_shot_example_1, few_shot_example_2, few_shot_example_3]

# Agent



In [21]:
imagePath = "/content/screenshot.png"

In [ ]:
task = (
    "Find the cheapest flight from Amsterdam to Madrid, departure 22 May, return 28 May"
)

print("Generating plan...")
planning_response = client.chat.completions.create(
    model=CFG.model,
    temperature=0.0,
    messages=[
        {"role": "system", "content": planning_prompt},
        {"role": "user", "content": task},
    ],
)

plan = planning_response.choices[0].message.content
print(plan)
steps = [line.strip()[3:] for line in plan.strip().split("\n")]

Generating plan...
1. Navigate to a flight aggregator website (e.g., Kayak).
2. Enter Amsterdam as the departure city.
3. Enter Madrid as the destination city.
4. Enter 22 May as the departure date.
5. Enter 28 May as the return date.
6. Search for flights.
7. Extract and compare the flight options, highlighting the cheapest option.


In [ ]:
previous_context = None


async def run_browser():
    async with async_playwright() as playwright:
        # Launch Chromium browser
        browser = await playwright.chromium.launch(headless=False, channel="chrome")
        page = await browser.new_page()
        await asyncio.sleep(1)
        await page.goto("https://google.com/")
        previous_actions = []
        try:
            while True:  # Infinite loop to keep session alive, press enter to continue or 'q' to quit
                # Get Context from page
                accessibility_tree = await page.accessibility.snapshot()
                accessibility_tree = parse_accessibility_tree(accessibility_tree)
                await page.screenshot(path="screenshot.png")
                base64_image = encode_image(imagePath)
                previous_context = accessibility_tree
                response = client.chat.completions.create(
                    model=CFG.model,
                    temperature=0.0,
                    messages=[
                        {"role": "system", "content": execution_prompt},
                        {
                            "role": "system",
                            "content": f"Few shot examples: {few_shot_examples}. Just a few examples, user will assign you VERY range set of tasks.",
                        },
                        {
                            "role": "system",
                            "content": f"Plan to execute: {steps}\n\n Accessibility Tree: {previous_context}\n\n, previous actions: {previous_actions}",
                        },
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": f"What should be the next action to accomplish the task: {task} based on the current state? Remember to review the plan and select the next action based on the current state. Provide the next action in JSON format strictly as specified above.",
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": f"data:image/jpeg;base64,{base64_image}",
                                    },
                                },
                            ],
                        },
                    ],
                )
                res = response.choices[0].message.content
                ## to remove invisible characters, whitespaces and commas:
                # Remove any trailing commas
                res = res.rstrip(",")
                # Remove any invisible characters
                res = "".join(
                    c for c in res if ord(c) >= 32 or ord(c) == 10 or ord(c) == 13
                )
                print("Agent response:", res)
                try:
                    match = re.search(r"\{.*\}", res, re.DOTALL)
                    if match:
                        output = json.loads(match.group(0))
                except Exception as e:
                    print("Error parsing JSON:", e)

                if output["action"] == "navigation":
                    try:
                        await page.goto(output["url"])
                        previous_actions.append(
                            f"navigated to {output['url']}, SUCCESS"
                        )
                    except Exception as e:
                        previous_actions.append(
                            f"Error navigating to {output['url']}: {e}"
                        )

                elif output["action"] == "click":
                    try:
                        selector_type, selector_name = (
                            output["selector"].split("=")[0],
                            output["selector"].split("=")[1],
                        )
                        res = await page.get_by_role(
                            selector_type, name=selector_name
                        ).first.click()
                        previous_actions.append(
                            f"clicked {output['selector']}, SUCCESS"
                        )
                    except Exception as e:
                        previous_actions.append(
                            f"Error clicking on {output['selector']}: {e}"
                        )

                elif (
                    output["action"] == "fill"
                    and output["selector"] == "textbox=outband_date"
                ):
                    try:
                        # Simulate a click to open the date picker if necessary
                        await page.click("button=outband_date")
                        await fill_date(
                            page, 'input[name="outband_date"]', output["value"]
                        )
                        previous_actions.append(
                            f"filled Departure date field with {output['value']}, SUCCESS"
                        )
                    except Exception as e:
                        previous_actions.append(
                            f"Error filling Departure date field with {output['value']}: {e}"
                        )
                elif (
                    output["action"] == "fill"
                    and output["selector"] == "textbox=return_date"
                ):
                    try:
                        # Simulate a click to open the date picker if necessary
                        await page.click("button=return_date")
                        await fill_date(
                            page, 'input[name="return_date"]', output["value"]
                        )
                        previous_actions.append(
                            f"filled Return date field with {output['value']}, SUCCESS"
                        )
                    except Exception as e:
                        previous_actions.append(
                            f"Error filling Return date field with {output['value']}: {e}"
                        )

                elif output["action"] == "fill":
                    try:
                        selector_type, selector_name = (
                            output["selector"].split("=")[0],
                            output["selector"].split("=")[1],
                        )
                        res = await page.get_by_role(
                            selector_type, name=selector_name
                        ).fill(output["value"])
                        await asyncio.sleep(1)
                        await page.keyboard.press("Enter")
                        previous_actions.append(
                            f"filled {output['selector']} with {output['value']}, SUCCESS"
                        )
                    except Exception as e:
                        previous_actions.append(
                            f"Error filling {output['selector']} with {output['value']}: {e}"
                        )

                elif output["action"] == "finished":
                    print(output["summary"])
                    break

                await asyncio.sleep(1)

                # Or wait for user input
                user_input = input("Press 'q' to quit or Enter to continue: ")
                if user_input.lower() == "q":
                    break

        except Exception as e:
            print(f"An error occurred: {e}")
        finally:
            # Only close the browser when explicitly requested
            await browser.close()

Ten kod definiuje asynchroniczną funkcję `run_browser`, która uruchamia przeglądarkę Chromium i automatyzuje interakcje z nią w celu wykonania zadania określonego przez zmienną `task`. Funkcja ta wykorzystuje biblioteki Playwright, Together AI oraz inne moduły importowane wcześniej.

**Główne elementy kodu:**

1.  **Inicjalizacja przeglądarki:**
    *   `async with async_playwright() as playwright:` – Uruchamia kontekst menedżera Playwright, zapewniając poprawne zarządzanie zasobami przeglądarki.
    *   `browser = await playwright.chromium.launch(headless=True, channel="chrome")` – Uruchamia przeglądarkę Chromium w trybie bezinterfejsu graficznego (`headless=True`). `channel="chrome"` określa wersję kanału Chrome do użycia.
    *   `page = await browser.new_page()` – Tworzy nową stronę przeglądarki.
    *   `await asyncio.sleep(1)` – Czeka 1 sekundę, aby strona się załadowała.
    *   `await page.goto("https://google.com/")` – Przechodzi na stronę Google.

2.  **Pętla główna interakcji:**
    *   `while True:` – Uruchamia nieskończoną pętlę, która kontynuuje działanie agenta do momentu zakończenia zadania lub ręcznego przerwania przez użytkownika.
    *   **Pozyskiwanie kontekstu:**
        *   `accessibility_tree = await page.accessibility.snapshot()` – Pobiera drzewo dostępności strony internetowej, które reprezentuje strukturę i zawartość strony w sposób zrozumiały dla agenta.
        *   `accessibility_tree = parse_accessibility_tree(accessibility_tree)` - Parsuje drzewo dostępności za pomocą funkcji `parse_accessibility_tree`.
        *   `await page.screenshot(path="screenshot.png")` – Robi zrzut ekranu strony i zapisuje go jako plik "screenshot.png".
        *   `base64_image = encode_image(imagePath)` - Koduje obraz do formatu base64 za pomocą funkcji `encode_image`.
    *   **Komunikacja z modelem językowym:**
        *   `response = client.chat.completions.create(...)` – Wysyła zapytanie do modelu językowego Together AI, przekazując:
            *   `model = CFG.model` – Nazwę używanego modelu.
            *   `temperature=0.0` – Ustawienie temperatury na 0.0 dla deterministycznych odpowiedzi.
            *   `messages=[...]` – Listę wiadomości, która zawiera:
                *   Wiadomość systemową z instrukcjami (`execution_prompt`).
                *   Przykłady interakcji (`few_shot_examples`).
                *   Plan działania (`steps`).
                *   Kontekst strony (drzewo dostępności `previous_context`).
                *   Poprzednie akcje (`previous_actions`).
                *   Zapytanie użytkownika o następną akcję, zawierające zarówno tekstowe zapytanie, jak i zakodowany obraz zrzutu ekranu.
    *   **Parsowanie odpowiedzi modelu:**
        *   `res = response.choices[0].message.content` – Pobiera odpowiedź modelu językowego.
        *   `match = re.search(r'\{.*\}', res, re.DOTALL)` – Szuka w odpowiedzi fragmentu JSON.
        *   `output = json.loads(match.group(0))` – Parsuje znaleziony fragment JSON do słownika Pythona.

3.  **Wykonanie akcji:**
    *   Instrukcje `if/elif` sprawdzają wartość klucza `"action"` w słowniku `output` i wykonują odpowiednią akcję:
        *   `"navigation"` – Przechodzi na podany adres URL (`output["url"]`).
        *   `"click"` – Klika element o określonym selektorze (`output["selector"]`).
        *   `"fill"` – Wypełnia pole tekstowe o określonym selektorze wartością z `output["value"]`.
        *   `"finished"` – Wyświetla podsumowanie zadania (`output["summary"]`) i przerywa pętlę.

4.  **Obsługa błędów:**
    *   Bloki `try/except` obsługują potencjalne błędy podczas wykonywania akcji, rejestrując je w liście `previous_actions`.

5.  **Interakcja z użytkownikiem:**
    *   `user_input = input("Press 'q' to quit or Enter to continue: ")` – Czeka na wprowadzenie danych przez użytkownika (naciśnięcie "Enter" aby kontynuować lub "q" aby zakończyć).

6.  **Zamykanie przeglądarki:**
    *   `finally:` – Blok `finally` zapewnia, że przeglądarka zostanie zamknięta nawet w przypadku wystąpienia błędów.
    *   `await browser.close()` – Zamyka przeglądarkę Chromium.

Podsumowując, ten kod implementuje agenta automatyzującego interakcje z przeglądarką internetową na podstawie planu działania generowanego przez model językowy. Agent pobiera kontekst strony, wysyła zapytania do modelu, wykonuje akcje zgodnie z otrzymanymi instrukcjami i powtarza proces w pętli, aż do zakończenia zadania lub ręcznego przerwania przez użytkownika.

In [ ]:
# Run the async function
await run_browser()

Agent response: {
    "current_state": "On Google's cookie consent page.",
    "reasoning": "The task is to find the cheapest flight from Amsterdam to Madrid, but currently, I am on Google's cookie consent page. I need to navigate to a flight aggregator website. Since I am currently on Google, I should first accept or reject the cookies to proceed.",
    "action": "click",
    "selector": "button=Alles afwijzen"
}
Press 'q' to quit or Enter to continue: 
Agent response: {
    "current_state": "On the Google homepage.",
    "reasoning": "The task is to find the cheapest flight from Amsterdam to Madrid, with departure on 22 May and return on 28 May. The first step is to navigate to a flight aggregator website. Since the current page is Google's homepage, I will use it to search for a flight aggregator website.",
    "action": "fill",
    "selector": "combobox=Zoek",
    "value": "Kayak"
}
Press 'q' to quit or Enter to continue: 
Agent response: {
    "current_state": "On a Google reCAPTC